# Single-Factor Vasicek Model

We first calibrate and simulate a univariate Vasicek model for the 6-month US Treasury yield.

The Vasicek model is:

$$
dr_t = \kappa(\theta-r_t)dt+\sigma dW_t
$$

where:

- $\kappa$ is the speed of mean reversion
- $\theta$ is the long-run mean
- $\sigma$ is the instantaneous annual volatility
- $r_t$ is the yield at time $t$    

In [ ]:
#Import the libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import PercentFormatter
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.stats import probplot

In [ ]:
df = pd.read_csv("../data/processed/cleaned_yield_curve.csv")

df.head()

Now we have a look at only the 6 month maturities, since this is a single factor model.

In [ ]:
vasicek_6m = df[
    ["Date", "6M"]
].copy()

vasicek_6m.head()

In [ ]:
vasicek_6m["Date"] = pd.to_datetime(
    vasicek_6m["Date"],
    errors="coerce"
)

vasicek_6m["6M"] = pd.to_numeric(
    vasicek_6m["6M"],
    errors="coerce"
)

vasicek_6m = (
    vasicek_6m
    .dropna()
    .sort_values("Date")
    .drop_duplicates(subset="Date")
    .reset_index(drop=True)
)

vasicek_6m.head()

In [ ]:
print(
    "Number of observations:",
    len(vasicek_6m)
)

print(
    "Starting date:",
    vasicek_6m["Date"].min()
)

print(
    "Ending date:",
    vasicek_6m["Date"].max()
)

print(
    "Missing values:"
)

print(
    vasicek_6m.isna().sum()
)

In [ ]:
print(
    "Duplicated dates:",
    vasicek_6m["Date"].duplicated().sum()
)

In [ ]:
vasicek_6m["Rate"] = (
    vasicek_6m["6M"] / 100
)

vasicek_6m.head()

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    vasicek_6m["Date"],
    vasicek_6m["Rate"],
    linewidth=1
)

plt.title(
    "Historical 6-Month US Treasury Yield"
)

plt.xlabel("Date")
plt.ylabel("6M yield")

plt.gca().yaxis.set_major_formatter(
    PercentFormatter(xmax=1)
)

plt.grid(alpha=0.3)
plt.show()

In [ ]:
vasicek_6m["Rate"]

In [ ]:
vasicek_6m["Next_Date"] = (
    vasicek_6m["Date"].shift(-1)
)

vasicek_6m["Next_Rate"] = (
    vasicek_6m["Rate"].shift(-1)
)

vasicek_6m.head()

In [ ]:
vasicek_6m["Calendar_Days"] = (
    vasicek_6m["Next_Date"]
    - vasicek_6m["Date"]
).dt.days

vasicek_6m[
    [
        "Date",
        "Next_Date",
        "Calendar_Days"
    ]
].head(10)

In [ ]:
vasicek_6m["Delta_t"] = (
    vasicek_6m["Calendar_Days"] / 365.25
)

In [ ]:
transitions_6m = vasicek_6m[
    [
        "Date",
        "Next_Date",
        "Rate",
        "Next_Rate",
        "Calendar_Days",
        "Delta_t"
    ]
].dropna().copy()

transitions_6m = transitions_6m[
    transitions_6m["Delta_t"] > 0
].reset_index(drop=True)

transitions_6m.head()

In [ ]:
transitions_6m = transitions_6m.rename(
    columns={
        "Rate": "r_t",
        "Next_Rate": "r_next"
    }
)

transitions_6m.head()

In [ ]:
print(
    "Number of transitions:",
    len(transitions_6m)
)

print(
    "Minimum time step:",
    transitions_6m["Delta_t"].min()
)

print(
    "Maximum time step:",
    transitions_6m["Delta_t"].max()
)

print(
    "Missing values:"
)

print(
    transitions_6m.isna().sum()
)

In [ ]:
transitions_6m[
    "Calendar_Days"
].value_counts().sort_index().head(15)

In [ ]:
transitions_6m

In [ ]:
#Important columns

transitions_6m["r_t"]
transitions_6m["r_next"]
transitions_6m["Delta_t"]

In [ ]:
r_t = transitions_6m["r_t"].to_numpy(
    dtype=float
)

r_next = transitions_6m["r_next"].to_numpy(
    dtype=float
)

delta_t = transitions_6m["Delta_t"].to_numpy(
    dtype=float
)

print("Number of transitions:", len(r_t))
print("First current rate:", r_t[0])
print("First next rate:", r_next[0])
print("First time step:", delta_t[0])

In [ ]:
def vasicek_negative_log_likelihood(
    parameters,
    r_t,
    r_next,
    delta_t
):
    log_kappa, theta, log_sigma = parameters

    kappa = np.exp(log_kappa)
    sigma = np.exp(log_sigma)

    phi = np.exp(-kappa * delta_t)

    conditional_mean = (
        theta
        + (r_t - theta) * phi
    )

    conditional_variance = (
        sigma ** 2
        * (1 - phi ** 2)
        / (2 * kappa)
    )

    if np.any(conditional_variance <= 0):
        return np.inf

    residuals = (
        r_next - conditional_mean
    )

    log_likelihood_terms = (
        -0.5
        * (
            np.log(
                2
                * np.pi
                * conditional_variance
            )
            +
            residuals ** 2
            / conditional_variance
        )
    )

    total_log_likelihood = np.sum(
        log_likelihood_terms
    )

    return -total_log_likelihood

In [ ]:
theta_initial = r_t.mean()

kappa_initial = 0.50

sigma_initial = np.sqrt(
    np.sum((r_next - r_t) ** 2)
    /
    np.sum(delta_t)
)

print(
    "Initial kappa:",
    kappa_initial
)

print(
    "Initial theta:",
    theta_initial
)

print(
    "Initial sigma:",
    sigma_initial
)

In [ ]:
initial_parameters = np.array([
    np.log(kappa_initial),
    theta_initial,
    np.log(sigma_initial)
])

initial_parameters

In [ ]:
#Now we run the MLE

mle_result_6m = minimize(
    vasicek_negative_log_likelihood,
    x0=initial_parameters,
    args=(
        r_t,
        r_next,
        delta_t
    ),
    method="L-BFGS-B",
    bounds=[
        (
            np.log(0.000001),
            np.log(10)
        ),
        (
            -0.10,
            0.20
        ),
        (
            np.log(0.000001),
            np.log(1)
        )
    ]
)

In [ ]:
print(
    "Optimization successful:",
    mle_result_6m.success
)

print(
    "Optimizer message:",
    mle_result_6m.message
)

print(
    "Negative log-likelihood:",
    mle_result_6m.fun
)

In [ ]:
log_kappa_6m, theta_6m, log_sigma_6m = (
    mle_result_6m.x
)

kappa_6m = np.exp(
    log_kappa_6m
)

sigma_6m = np.exp(
    log_sigma_6m
)

In [ ]:
print(
    f"Kappa: {kappa_6m:.6f}"
)

print(
    f"Theta: {theta_6m:.4%}"
)

print(
    f"Sigma: {sigma_6m:.4%}"
)

In [ ]:
half_life_years_6m = (
    np.log(2)
    /
    kappa_6m
)

print(
    f"Half-life: "
    f"{half_life_years_6m:.2f} years"
)

In [ ]:
parameters_6m = pd.Series({
    "Maturity": "6M",
    "Kappa": kappa_6m,
    "Theta": theta_6m,
    "Theta_Percent": theta_6m * 100,
    "Sigma": sigma_6m,
    "Sigma_Percent": sigma_6m * 100,
    "Half_Life_Years":
        half_life_years_6m,
    "Log_Likelihood":
        -mle_result_6m.fun,
    "Optimization_Success":
        mle_result_6m.success
})

parameters_6m

In [ ]:
print(
    "Kappa at boundary:",
    np.isclose(kappa_6m, 0.000001)
    or np.isclose(kappa_6m, 10)
)

print(
    "Theta at boundary:",
    np.isclose(theta_6m, -0.10)
    or np.isclose(theta_6m, 0.20)
)

print(
    "Sigma at boundary:",
    np.isclose(sigma_6m, 0.000001)
    or np.isclose(sigma_6m, 1)
)

In [ ]:
phi_6m = np.exp(
    -kappa_6m * delta_t
)

fitted_mean_6m = (
    theta_6m
    + (r_t - theta_6m) * phi_6m
)

In [ ]:
fitted_variance_6m = (
    sigma_6m ** 2
    * (1 - phi_6m ** 2)
    / (2 * kappa_6m)
)

fitted_sd_6m = np.sqrt(
    fitted_variance_6m
)

In [ ]:
residuals_6m = (
    r_next - fitted_mean_6m
)

standardized_residuals_6m = (
    residuals_6m
    / fitted_sd_6m
)

In [ ]:
validation_6m = transitions_6m.copy()

validation_6m[
    "Fitted_Mean"
] = fitted_mean_6m

validation_6m[
    "Fitted_SD"
] = fitted_sd_6m

validation_6m[
    "Residual"
] = residuals_6m

validation_6m[
    "Standardized_Residual"
] = standardized_residuals_6m

validation_6m.head()

In [ ]:
print(
    "Residual mean:",
    validation_6m[
        "Residual"
    ].mean()
)

print(
    "Standardized residual mean:",
    validation_6m[
        "Standardized_Residual"
    ].mean()
)

print(
    "Standardized residual SD:",
    validation_6m[
        "Standardized_Residual"
    ].std(ddof=0)
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    validation_6m["Next_Date"],
    validation_6m[
        "Standardized_Residual"
    ],
    linewidth=0.7
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.axhline(
    2,
    color="red",
    linestyle=":",
    alpha=0.7
)

plt.axhline(
    -2,
    color="red",
    linestyle=":",
    alpha=0.7
)

plt.title(
    "6M Vasicek Standardized Residuals"
)

plt.xlabel("Date")
plt.ylabel("Standardized residual")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    standardized_residuals_6m,
    bins=60,
    density=True,
    alpha=0.7,
    edgecolor="white",
    label="Observed residuals"
)

x = np.linspace(
    -4,
    4,
    500
)

plt.plot(
    x,
    norm.pdf(x),
    color="red",
    linewidth=2,
    label="Standard normal"
)

plt.title(
    "Distribution of 6M Standardized Residuals"
)

plt.xlabel("Standardized residual")
plt.ylabel("Density")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))

probplot(
    standardized_residuals_6m,
    dist="norm",
    plot=plt
)

plt.title(
    "Q–Q Plot of 6M Standardized Residuals"
)

plt.grid(alpha=0.3)
plt.show()

## Step 2: Calculate the Exact One-Step Vasicek Quantities

For a time step $\Delta t$, the exact Vasicek transition is:

$$
r_{t+\Delta t}
=
\theta
+
(r_t-\theta)e^{-\kappa \Delta t}
+
\sigma
\sqrt{
\frac{1-e^{-2\kappa \Delta t}}{2\kappa}
}
Z
$$

where:

$$
Z \sim \mathcal{N}(0,1)
$$

The conditional mean is:

$$
\mathbb{E}[r_{t+\Delta t}\mid r_t]
=
\theta+(r_t-\theta)e^{-\kappa\Delta t}
$$

The conditional variance is:

$$
\operatorname{Var}(r_{t+\Delta t}\mid r_t)
=
\frac{\sigma^2}{2\kappa}
\left(1-e^{-2\kappa\Delta t}\right)
$$


Now we make it into a function

In [ ]:
def fit_vasicek_mle(
    data,
    maturity,
    date_column="Date",
    rates_are_percent=True
):
    """
    Calibrate a single-factor Vasicek model for one maturity
    using exact maximum likelihood.
    """

    # Select and clean the required columns
    model_data = data[
        [date_column, maturity]
    ].copy()

    model_data[date_column] = pd.to_datetime(
        model_data[date_column],
        errors="coerce"
    )

    model_data[maturity] = pd.to_numeric(
        model_data[maturity],
        errors="coerce"
    )

    model_data = (
        model_data
        .dropna()
        .sort_values(date_column)
        .drop_duplicates(subset=date_column)
        .reset_index(drop=True)
    )

    # Convert percentage yields into decimals
    model_data["Rate"] = model_data[maturity]

    if rates_are_percent:
        model_data["Rate"] = (
            model_data["Rate"] / 100
        )

    # Construct consecutive transitions
    model_data["Next_Date"] = (
        model_data[date_column].shift(-1)
    )

    model_data["Next_Rate"] = (
        model_data["Rate"].shift(-1)
    )

    model_data["Calendar_Days"] = (
        model_data["Next_Date"]
        - model_data[date_column]
    ).dt.days

    model_data["Delta_t"] = (
        model_data["Calendar_Days"] / 365.25
    )

    transitions = model_data[
        [
            date_column,
            "Next_Date",
            "Rate",
            "Next_Rate",
            "Calendar_Days",
            "Delta_t"
        ]
    ].dropna().copy()

    transitions = transitions[
        transitions["Delta_t"] > 0
    ].reset_index(drop=True)

    transitions = transitions.rename(
        columns={
            "Rate": "r_t",
            "Next_Rate": "r_next"
        }
    )

    r_t = transitions["r_t"].to_numpy(
        dtype=float
    )

    r_next = transitions["r_next"].to_numpy(
        dtype=float
    )

    delta_t = transitions["Delta_t"].to_numpy(
        dtype=float
    )

    # Initial parameter values
    kappa_initial = 0.50
    theta_initial = r_t.mean()

    sigma_initial = np.sqrt(
        np.sum((r_next - r_t) ** 2)
        / np.sum(delta_t)
    )

    initial_parameters = np.array([
        np.log(kappa_initial),
        theta_initial,
        np.log(sigma_initial)
    ])

    # Numerical MLE
    mle_result = minimize(
        vasicek_negative_log_likelihood,
        x0=initial_parameters,
        args=(
            r_t,
            r_next,
            delta_t
        ),
        method="L-BFGS-B",
        bounds=[
            (
                np.log(0.000001),
                np.log(10)
            ),
            (
                -0.10,
                0.20
            ),
            (
                np.log(0.000001),
                np.log(1)
            )
        ]
    )

    log_kappa, theta, log_sigma = (
        mle_result.x
    )

    kappa = np.exp(log_kappa)
    sigma = np.exp(log_sigma)

    half_life_years = (
        np.log(2) / kappa
    )

    # Calculate fitted conditional distributions
    phi = np.exp(
        -kappa * delta_t
    )

    fitted_mean = (
        theta
        + (r_t - theta) * phi
    )

    fitted_variance = (
        sigma ** 2
        * (1 - phi ** 2)
        / (2 * kappa)
    )

    fitted_sd = np.sqrt(
        fitted_variance
    )

    residuals = (
        r_next - fitted_mean
    )

    standardized_residuals = (
        residuals / fitted_sd
    )

    transitions["Fitted_Mean"] = (
        fitted_mean
    )

    transitions["Fitted_SD"] = (
        fitted_sd
    )

    transitions["Residual"] = (
        residuals
    )

    transitions["Standardized_Residual"] = (
        standardized_residuals
    )

    parameter_results = {
        "Maturity": maturity,
        "Number_of_Observations": len(model_data),
        "Number_of_Transitions": len(transitions),
        "Kappa": kappa,
        "Theta": theta,
        "Theta_Percent": theta * 100,
        "Sigma": sigma,
        "Sigma_Percent": sigma * 100,
        "Half_Life_Years": half_life_years,
        "Log_Likelihood": -mle_result.fun,
        "Optimization_Success": mle_result.success,
        "Optimizer_Message": mle_result.message,
        "Latest_Rate": model_data["Rate"].iloc[-1],
        "Latest_Date": model_data[date_column].iloc[-1],
        "Residual_Mean": residuals.mean(),
        "Standardized_Residual_Mean":
            standardized_residuals.mean(),
        "Standardized_Residual_SD":
            standardized_residuals.std(ddof=0)
    }

    return (
        parameter_results,
        transitions
    )

In [ ]:
parameters_6m_function, validation_6m_function = (
    fit_vasicek_mle(
        data=df,
        maturity="6M",
        date_column="Date",
        rates_are_percent=True
    )
)

pd.Series(parameters_6m_function)

In [ ]:
maturities = [
    "6M",
    "1Y",
    "2Y",
    "3Y",
    "5Y",
    "7Y",
    "10Y"
]

In [ ]:
all_parameter_results = []

all_validation_results = {}

for maturity in maturities:

    parameters, validation = (
        fit_vasicek_mle(
            data=df,
            maturity=maturity,
            date_column="Date",
            rates_are_percent=True
        )
    )

    all_parameter_results.append(
        parameters
    )

    all_validation_results[maturity] = (
        validation
    )

In [ ]:
vasicek_parameter_table = pd.DataFrame(
    all_parameter_results
)

vasicek_parameter_table[
    [
        "Maturity",
        "Number_of_Observations",
        "Kappa",
        "Theta_Percent",
        "Sigma_Percent",
        "Half_Life_Years",
        "Log_Likelihood",
        "Optimization_Success"
    ]
]

In [ ]:
display_parameter_table = (
    vasicek_parameter_table[
        [
            "Maturity",
            "Number_of_Observations",
            "Kappa",
            "Theta_Percent",
            "Sigma_Percent",
            "Half_Life_Years",
            "Log_Likelihood",
            "Optimization_Success"
        ]
    ]
    .copy()
)

display_parameter_table[
    [
        "Kappa",
        "Theta_Percent",
        "Sigma_Percent",
        "Half_Life_Years",
        "Log_Likelihood"
    ]
] = display_parameter_table[
    [
        "Kappa",
        "Theta_Percent",
        "Sigma_Percent",
        "Half_Life_Years",
        "Log_Likelihood"
    ]
].round(4)

display_parameter_table

In [ ]:
vasicek_parameter_table[
    [
        "Maturity",
        "Optimization_Success",
        "Optimizer_Message"
    ]
]

In [ ]:
vasicek_parameter_table[
    [
        "Maturity",
        "Residual_Mean",
        "Standardized_Residual_Mean",
        "Standardized_Residual_SD"
    ]
]

In [ ]:
all_validation_results["1Y"].head()

In [ ]:
all_validation_results["10Y"].head()

In [ ]:
probplot(
    all_validation_results["10Y"][
        "Standardized_Residual"
    ],
    dist="norm",
    plot=plt
)

plt.title(
    "Q–Q Plot of 10Y Vasicek Standardized Residuals"
)

plt.grid(alpha=0.3)
plt.show()

In [ ]:
def simulate_vasicek(
    kappa,
    theta,
    sigma,
    initial_rate,
    simulation_horizon_years=1,
    steps_per_year=252,
    number_of_simulations=10000,
    seed=42
):
    """
    Simulate a single-factor Vasicek process using
    the exact conditional transition distribution.
    """

    number_of_steps = int(
        simulation_horizon_years
        * steps_per_year
    )

    delta_t = 1 / steps_per_year

    # Exact one-step Vasicek quantities
    phi = np.exp(
        -kappa * delta_t
    )

    conditional_sd = np.sqrt(
        sigma ** 2
        * (1 - phi ** 2)
        / (2 * kappa)
    )

    random_generator = np.random.default_rng(
        seed=seed
    )

    simulated_paths = np.empty(
        (
            number_of_steps + 1,
            number_of_simulations
        )
    )

    simulated_paths[0, :] = initial_rate

    for step in range(
        1,
        number_of_steps + 1
    ):

        random_shocks = random_generator.normal(
            loc=0,
            scale=1,
            size=number_of_simulations
        )

        simulated_paths[step, :] = (
            theta
            + (
                simulated_paths[step - 1, :]
                - theta
            ) * phi
            + conditional_sd * random_shocks
        )

    simulation_times = np.linspace(
        0,
        simulation_horizon_years,
        number_of_steps + 1
    )

    simulated_paths = pd.DataFrame(
        simulated_paths,
        index=simulation_times
    )

    simulated_paths.index.name = (
        "Time_In_Years"
    )

    terminal_rates = (
        simulated_paths.iloc[-1].copy()
    )

    simulated_changes = (
        terminal_rates - initial_rate
    )

    return {
        "Paths": simulated_paths,
        "Terminal_Rates": terminal_rates,
        "Simulated_Changes": simulated_changes,
        "Phi": phi,
        "Conditional_SD": conditional_sd
    }

In [ ]:
parameter_row_6m = (
    vasicek_parameter_table
    .loc[
        vasicek_parameter_table["Maturity"] == "6M"
    ]
    .iloc[0]
)

In [ ]:
simulation_6m_function = simulate_vasicek(
    kappa=parameter_row_6m["Kappa"],
    theta=parameter_row_6m["Theta"],
    sigma=parameter_row_6m["Sigma"],
    initial_rate=parameter_row_6m["Latest_Rate"],
    simulation_horizon_years=1,
    steps_per_year=252,
    number_of_simulations=10000,
    seed=42
)

In [ ]:
simulation_6m_function["Paths"].head()

In [ ]:
simulation_6m_function["Terminal_Rates"].describe()

In [ ]:
all_simulation_results = {}

for maturity_number, maturity in enumerate(
    maturities
):

    parameter_row = (
        vasicek_parameter_table
        .loc[
            vasicek_parameter_table[
                "Maturity"
            ] == maturity
        ]
        .iloc[0]
    )

    simulation_result = simulate_vasicek(
        kappa=parameter_row["Kappa"],
        theta=parameter_row["Theta"],
        sigma=parameter_row["Sigma"],
        initial_rate=parameter_row["Latest_Rate"],
        simulation_horizon_years=1,
        steps_per_year=252,
        number_of_simulations=10000,
        seed=42 + maturity_number
    )

    all_simulation_results[maturity] = (
        simulation_result
    )

In [ ]:
terminal_summary_rows = []

for maturity in maturities:

    parameter_row = (
        vasicek_parameter_table
        .loc[
            vasicek_parameter_table[
                "Maturity"
            ] == maturity
        ]
        .iloc[0]
    )

    terminal_rates = (
        all_simulation_results[maturity][
            "Terminal_Rates"
        ]
    )

    terminal_summary_rows.append({

        "Maturity":
            maturity,

        "Starting_Rate":
            parameter_row["Latest_Rate"],

        "Mean_After_1Y":
            terminal_rates.mean(),

        "Median_After_1Y":
            terminal_rates.median(),

        "Standard_Deviation":
            terminal_rates.std(ddof=1),

        "0.5th_Percentile":
            terminal_rates.quantile(0.005),

        "99.5th_Percentile":
            terminal_rates.quantile(0.995)
    })

In [ ]:
terminal_yield_summary = pd.DataFrame(
    terminal_summary_rows
)

terminal_yield_summary

In [ ]:
terminal_yield_summary_display = (
    terminal_yield_summary.copy()
)

percentage_columns = [
    "Starting_Rate",
    "Mean_After_1Y",
    "Median_After_1Y",
    "Standard_Deviation",
    "0.5th_Percentile",
    "99.5th_Percentile"
]

terminal_yield_summary_display[
    percentage_columns
] = (
    terminal_yield_summary_display[
        percentage_columns
    ] * 100
).round(4)

terminal_yield_summary_display

In [ ]:
change_summary_rows = []

for maturity in maturities:

    simulated_changes = (
        all_simulation_results[maturity][
            "Simulated_Changes"
        ]
    )

    change_summary_rows.append({

        "Maturity":
            maturity,

        "Mean_Change":
            simulated_changes.mean(),

        "Standard_Deviation":
            simulated_changes.std(ddof=1),

        "0.5th_Percentile":
            simulated_changes.quantile(0.005),

        "Median":
            simulated_changes.median(),

        "99.5th_Percentile":
            simulated_changes.quantile(0.995)
    })

In [ ]:
simulated_change_summary = pd.DataFrame(
    change_summary_rows
)

simulated_change_summary_bps = (
    simulated_change_summary.copy()
)

change_columns = [
    "Mean_Change",
    "Standard_Deviation",
    "0.5th_Percentile",
    "Median",
    "99.5th_Percentile"
]

simulated_change_summary_bps[
    change_columns
] = (
    simulated_change_summary_bps[
        change_columns
    ] * 10000
).round(2)

simulated_change_summary_bps

In [ ]:
fig, axes = plt.subplots(
    4,
    2,
    figsize=(16, 18),
    sharex=True
)

axes = axes.flatten()

for index, maturity in enumerate(
    maturities
):

    ax = axes[index]

    paths = (
        all_simulation_results[maturity][
            "Paths"
        ]
    )

    parameter_row = (
        vasicek_parameter_table
        .loc[
            vasicek_parameter_table[
                "Maturity"
            ] == maturity
        ]
        .iloc[0]
    )

    ax.plot(
        paths.index,
        paths.iloc[:, :100],
        linewidth=0.5,
        alpha=0.25
    )

    ax.axhline(
        parameter_row["Theta"],
        color="black",
        linestyle="--",
        linewidth=2,
        label="Long-run mean"
    )

    ax.axhline(
        parameter_row["Latest_Rate"],
        color="red",
        linestyle=":",
        linewidth=2,
        label="Starting rate"
    )

    ax.set_title(
        f"{maturity} Vasicek Simulated Paths"
    )

    ax.set_xlabel("Time in years")
    ax.set_ylabel("Yield")

    ax.yaxis.set_major_formatter(
        PercentFormatter(xmax=1)
    )

    ax.grid(alpha=0.3)
    ax.legend()

fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

In [ ]:
#Now we calculate historical one-year yield changes

def calculate_historical_one_year_changes(
    data,
    maturity,
    date_column="Date",
    rates_are_percent=True,
    tolerance_days=7
):
    historical_data = data[
        [date_column, maturity]
    ].copy()

    historical_data[date_column] = pd.to_datetime(
        historical_data[date_column],
        errors="coerce"
    )

    historical_data[maturity] = pd.to_numeric(
        historical_data[maturity],
        errors="coerce"
    )

    historical_data = (
        historical_data
        .dropna()
        .sort_values(date_column)
        .drop_duplicates(subset=date_column)
        .reset_index(drop=True)
    )

    historical_data["Rate"] = (
        historical_data[maturity]
    )

    if rates_are_percent:
        historical_data["Rate"] = (
            historical_data["Rate"] / 100
        )

    # Target date exactly one calendar year later
    starting_observations = historical_data[
        [date_column, "Rate"]
    ].copy()

    starting_observations["Target_Date"] = (
        starting_observations[date_column]
        + pd.DateOffset(years=1)
    )

    future_observations = historical_data[
        [date_column, "Rate"]
    ].rename(
        columns={
            date_column: "Future_Date",
            "Rate": "Future_Rate"
        }
    )

    matched_data = pd.merge_asof(
        starting_observations.sort_values(
            "Target_Date"
        ),
        future_observations.sort_values(
            "Future_Date"
        ),
        left_on="Target_Date",
        right_on="Future_Date",
        direction="nearest",
        tolerance=pd.Timedelta(
            days=tolerance_days
        )
    )

    matched_data = (
        matched_data
        .dropna(subset=["Future_Rate"])
        .reset_index(drop=True)
    )

    matched_data["Actual_Days"] = (
        matched_data["Future_Date"]
        - matched_data[date_column]
    ).dt.days

    matched_data["Historical_1Y_Change"] = (
        matched_data["Future_Rate"]
        - matched_data["Rate"]
    )

    matched_data["Maturity"] = maturity

    return matched_data


In [ ]:
all_historical_change_results = {}

for maturity in maturities:

    historical_changes = (
        calculate_historical_one_year_changes(
            data=df,
            maturity=maturity,
            date_column="Date",
            rates_are_percent=True,
            tolerance_days=7
        )
    )

    all_historical_change_results[maturity] = (
        historical_changes
    )

In [ ]:
all_historical_change_results["6M"].head()

In [ ]:
all_historical_change_results["6M"][
    "Actual_Days"
].describe()

In [ ]:
historical_summary_rows = []

for maturity in maturities:

    historical_changes = (
        all_historical_change_results[maturity][
            "Historical_1Y_Change"
        ]
    )

    historical_summary_rows.append({

        "Maturity":
            maturity,

        "Number_of_Changes":
            historical_changes.count(),

        "Mean_Change":
            historical_changes.mean(),

        "Standard_Deviation":
            historical_changes.std(ddof=1),

        "0.5th_Percentile":
            historical_changes.quantile(0.005),

        "Median":
            historical_changes.median(),

        "99.5th_Percentile":
            historical_changes.quantile(0.995)
    })

In [ ]:
historical_change_summary = pd.DataFrame(
    historical_summary_rows
)

historical_change_summary_bps = (
    historical_change_summary.copy()
)

historical_change_columns = [
    "Mean_Change",
    "Standard_Deviation",
    "0.5th_Percentile",
    "Median",
    "99.5th_Percentile"
]

historical_change_summary_bps[
    historical_change_columns
] = (
    historical_change_summary_bps[
        historical_change_columns
    ] * 10000
).round(2)

historical_change_summary_bps

In [ ]:
tail_comparison = (
    historical_change_summary[
        [
            "Maturity",
            "Number_of_Changes",
            "Standard_Deviation",
            "0.5th_Percentile",
            "99.5th_Percentile"
        ]
    ]
    .merge(
        simulated_change_summary[
            [
                "Maturity",
                "Standard_Deviation",
                "0.5th_Percentile",
                "99.5th_Percentile"
            ]
        ],
        on="Maturity",
        suffixes=(
            "_Historical",
            "_Vasicek"
        )
    )
)

In [ ]:
tail_comparison[
    "Lower_Tail_Difference"
] = (
    tail_comparison[
        "0.5th_Percentile_Vasicek"
    ]
    - tail_comparison[
        "0.5th_Percentile_Historical"
    ]
)

tail_comparison[
    "Upper_Tail_Difference"
] = (
    tail_comparison[
        "99.5th_Percentile_Vasicek"
    ]
    - tail_comparison[
        "99.5th_Percentile_Historical"
    ]
)

In [ ]:
tail_comparison_bps = (
    tail_comparison.copy()
)

comparison_rate_columns = [
    "Standard_Deviation_Historical",
    "Standard_Deviation_Vasicek",
    "0.5th_Percentile_Historical",
    "0.5th_Percentile_Vasicek",
    "99.5th_Percentile_Historical",
    "99.5th_Percentile_Vasicek",
    "Lower_Tail_Difference",
    "Upper_Tail_Difference"
]

tail_comparison_bps[
    comparison_rate_columns
] = (
    tail_comparison_bps[
        comparison_rate_columns
    ] * 10000
).round(2)

tail_comparison_bps

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 6)
)

x_positions = np.arange(
    len(maturities)
)

bar_width = 0.35

axes[0].bar(
    x_positions - bar_width / 2,
    tail_comparison_bps[
        "0.5th_Percentile_Historical"
    ],
    width=bar_width,
    label="Historical",
    color="steelblue"
)

axes[0].bar(
    x_positions + bar_width / 2,
    tail_comparison_bps[
        "0.5th_Percentile_Vasicek"
    ],
    width=bar_width,
    label="Vasicek",
    color="orange"
)

axes[0].set_title(
    "Historical vs Vasicek 0.5th Percentile"
)

axes[0].set_ylabel("One-year change in basis points")
axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(maturities)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend()

axes[1].bar(
    x_positions - bar_width / 2,
    tail_comparison_bps[
        "99.5th_Percentile_Historical"
    ],
    width=bar_width,
    label="Historical",
    color="steelblue"
)

axes[1].bar(
    x_positions + bar_width / 2,
    tail_comparison_bps[
        "99.5th_Percentile_Vasicek"
    ],
    width=bar_width,
    label="Vasicek",
    color="orange"
)

axes[1].set_title(
    "Historical vs Vasicek 99.5th Percentile"
)

axes[1].set_ylabel("One-year change in basis points")
axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(maturities)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()